In [2]:
import cv2 as cv
import pandas as pd
import easyocr
import re
import logging
import numpy as np
from pathlib import Path


# Configuração
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# Recortando o display
ROI = {"y1": 300, "y2": 400, "x1": 150, "x2": 450}

# Vídeos e suas amostras correspondentes
N_AMOSTRAS = 12
VIDEOS  = [f"video_{i}.mp4"                       for i in range(1, N_AMOSTRAS + 1)]
CSVS    = [f"VariacaoRelativa_Amostra{i}.csv"      for i in range(1, N_AMOSTRAS + 1)]

# Janelas de tempo para o valor esperado
JANELAS = [
    (0,  10,   0,   0),
    (10, 20, 100, 100),
    (20, 30, 200, 200),
    (30, 40, 300, 300),
    (40, 50, 400, 400),
    (50, 70, 500, 500),
]
MARGEM_G        = 80
SNAP_CANDIDATOS = [0, 100, 200, 300, 400, 500]

# Criando ocr
reader = easyocr.Reader(["en"], gpu=False)


# Definindo o valor esperado na janela de tempo
def valor_esperado(tempo: float) -> float:
    for t0, t1, g0, g1 in JANELAS:
        if t0 <= tempo < t1:
            frac = (tempo - t0) / (t1 - t0) if t1 > t0 else 0
            return g0 + frac * (g1 - g0)
    return JANELAS[-1][3]

# Estratégia de fallback - corrige para o valor esperado para casos onde o valor lido tem diferença maior que a margem estabelecida
def corrigir_com_contexto(valor_ocr: float, tempo: float) -> tuple[float, str]:
    esperado = valor_esperado(tempo)
    if abs(valor_ocr - esperado) <= MARGEM_G:
        return valor_ocr, "ocr_aceito"
    if valor_ocr == 0.0:
        return esperado, "fallback_esperado"
    melhor = min(SNAP_CANDIDATOS, key=lambda c: abs(c - esperado))
    if abs(melhor - esperado) <= MARGEM_G:
        return melhor, f"snap_{valor_ocr:.0f}→{melhor:.0f}"
    return esperado, f"fallback_esperado(ocr={valor_ocr:.0f})"

# Pré processamento do display para facilitar a leitura (várias estratégias)
def preprocessar(display: np.ndarray) -> list[tuple[str, np.ndarray]]:
    # Amplia a imagem para facilitar a leitura pelo OCR
    scale = 5
    h, w = display.shape[:2]
    grande = cv.resize(
        display,
        (w * scale, h * scale),
        interpolation=cv.INTER_CUBIC
    )

    # Extrai apenas o canal vermelho
    r = grande[:, :, 2]

    # Binarização com limiar fixo
    _, thresh_r80 = cv.threshold(r, 80, 255, cv.THRESH_BINARY)

    # Binarização automática usando o método de Otsu
    _, thresh_rotsu = cv.threshold(
        r, 0, 255,
        cv.THRESH_BINARY + cv.THRESH_OTSU
    )

    # Kernel usado para operações morfológicas
    kernel = np.ones((2, 2), np.uint8)

    # Dilatação para engrossar os dígitos
    dilated = cv.dilate(thresh_r80, kernel, iterations=1)
    dilated_otsu = cv.dilate(thresh_rotsu, kernel, iterations=1)

    # Conversão para HSV para isolar regiões vermelhas
    hsv = cv.cvtColor(grande, cv.COLOR_BGR2HSV)

    # Faixas de vermelho no HSV
    mask1 = cv.inRange(hsv, (0, 60, 80), (20, 255, 255))
    mask2 = cv.inRange(hsv, (160, 60, 80), (180, 255, 255))

    # Une as máscaras e dilata o resultado
    dilated_hsv = cv.dilate(
        cv.bitwise_or(mask1, mask2),
        kernel,
        iterations=1
    )

    # Converte para escala de cinza
    gray = cv.cvtColor(grande, cv.COLOR_BGR2GRAY)

    # Aumenta o contraste local da imagem
    clahe = cv.createCLAHE(
        clipLimit=3.0,
        tileGridSize=(8, 8)
    )
    eq = clahe.apply(gray)

    # Binarização automática após melhoria de contraste
    _, otsu_gray = cv.threshold(
        eq, 0, 255,
        cv.THRESH_BINARY + cv.THRESH_OTSU
    )

    # Retorna várias versões da imagem para testar no OCR
    return [
        ("r_thresh80_dilated", dilated),      # vermelho + threshold fixo + dilatação
        ("r_otsu_dilated", dilated_otsu),     # vermelho + Otsu + dilatação
        ("r_thresh80", thresh_r80),           # vermelho + threshold fixo
        ("hsv_dilated", dilated_hsv),         # máscara HSV do vermelho
        ("r_otsu", thresh_rotsu),             # vermelho + Otsu
        ("gray_otsu", otsu_gray),             # cinza + CLAHE + Otsu
        ("r_raw", r),                         # canal vermelho original
    ]

# OCR 
def extrair_valor_ocr(imagens: list[tuple[str, np.ndarray]]) -> tuple[float, str]:
    # Percorre todas as versões pré-processadas da imagem
    for nome, img in imagens:

        # Executa OCR permitindo apenas números e ponto decimal
        resultados = reader.readtext(
            img,
            allowlist="0123456789.",
            detail=1
        )

        # Analisa cada texto encontrado pelo OCR
        for _, texto, confianca in resultados:

            # Ignora leituras com baixa confiança
            if confianca < 0.25:
                continue

            # Procura um número no texto reconhecido
            match = re.search(r"\d+\.?\d*", texto)

            if match:
                try:
                    # Converte o número encontrado para float
                    # e retorna junto com o nome do pré-processamento
                    return float(match.group()), nome

                except ValueError:
                    # Caso a conversão falhe, tenta o próximo resultado
                    continue

    # Se nenhum valor válido for encontrado
    return 0.0, "nenhuma"

# Lendo um tempo no vídeo
def ler_massa_em_tempo(cap, fps: float, tempo: float, roi: dict) -> tuple[float, float, str, str]:

    # Coordenadas da região do visor (ROI)
    y1, y2 = roi["y1"], roi["y2"]
    x1, x2 = roi["x1"], roi["x2"]

    # Converte o tempo desejado para número do frame
    frame_alvo = int(round(tempo * fps))

    # Posiciona o vídeo no frame correspondente
    cap.set(cv.CAP_PROP_POS_FRAMES, frame_alvo)
    ret, frame = cap.read()

    # Se não conseguir ler o frame, usa valor esperado
    if not ret:
        esperado = valor_esperado(tempo)
        return 0.0, esperado, "erro_leitura", "fallback_esperado"

    # Recorta apenas a região do display
    display = frame[y1:y2, x1:x2]

    # Gera diferentes versões da imagem
    imagens = preprocessar(display)

    # Tenta extrair um valor usando OCR
    valor_ocr, estrategia = extrair_valor_ocr(imagens)

    # Aplica correções usando conhecimento do experimento
    valor_final, correcao = corrigir_com_contexto(valor_ocr, tempo)

    return valor_ocr, valor_final, estrategia, correcao


# Processando o par vídeo-csv
def processar_amostra(
    video_path: str,
    csv_path: str,
    roi: dict,
    amostra_id: int,
) -> pd.DataFrame:

    # Carrega os dados experimentais
    df = pd.read_csv(csv_path)

    # Procura a coluna de tempo
    col_t = next((c for c in df.columns if c.strip().lower() == "t"), None)

    if col_t is None:
        raise ValueError(
            f"Coluna 't' não encontrada em {csv_path}. "
            f"Colunas: {list(df.columns)}"
        )

    # Abre o vídeo
    cap = cv.VideoCapture(video_path)

    if not cap.isOpened():
        raise FileNotFoundError(
            f"Não foi possível abrir o vídeo: {video_path}"
        )

    # Obtém a taxa de quadros do vídeo
    fps = cap.get(cv.CAP_PROP_FPS)

    log.info(
        "  Vídeo: %s | FPS: %.2f | Pontos: %d",
        video_path,
        fps,
        len(df)
    )

    # Listas para armazenar resultados
    tempos = df[col_t].tolist()

    valores_ocr = []      # leitura bruta do OCR
    valores_final = []    # valor após correções
    estrategias = []      # pré-processamento utilizado
    correcoes = []        # tipo de correção aplicada

    # Processa cada instante de tempo do CSV
    for i, tempo in enumerate(tempos, 1):

        v_ocr, v_final, estrat, corr = ler_massa_em_tempo(
            cap,
            fps,
            tempo,
            roi
        )

        valores_ocr.append(v_ocr)
        valores_final.append(v_final)
        estrategias.append(estrat)
        correcoes.append(corr)

        # Mostra progresso periodicamente
        if i % 20 == 0 or i == len(tempos):
            log.info(
                "  [%d/%d] t=%.2fs → ocr=%.1f | final=%.1f | %s",
                i,
                len(tempos),
                tempo,
                v_ocr,
                v_final,
                corr
            )

    # Fecha o vídeo
    cap.release()

    # Cria um novo DataFrame mantendo as colunas originais
    df_completo = df.copy()

    # Adiciona os resultados obtidos do vídeo
    df_completo["massa_g"] = valores_final
    df_completo["ocr_bruto"] = valores_ocr
    df_completo["estrategia"] = estrategias
    df_completo["correcao"] = correcoes

    return df_completo

# Loop principal
def processar_todas(
    videos: list  = VIDEOS,
    csvs: list    = CSVS,
    roi: dict     = ROI,
    output_dir: str = ".",
):
    out = Path(output_dir)
    out.mkdir(exist_ok=True)

    for i, (video_path, csv_path) in enumerate(zip(videos, csvs), 1):
        log.info("=== Amostra %d | Vídeo: %s | CSV: %s ===", i, video_path, csv_path)

        if not Path(video_path).exists():
            log.warning("Vídeo não encontrado, pulando: %s", video_path)
            continue
        if not Path(csv_path).exists():
            log.warning("CSV não encontrado, pulando: %s", csv_path)
            continue

        try:
            df_completo = processar_amostra(video_path, csv_path, roi, i)
        except Exception as e:
            log.error("Erro ao processar amostra %d: %s", i, e)
            continue

        # CSV simples: t + massa
        path_massa = out / f"Massa_Amostra{i}.csv"
        df_completo[["t", "massa_g"]].to_csv(path_massa, index=False)

        # CSV completo: variacao relativa + massa + auditoria
        path_completo = out / f"Completo_Amostra{i}.csv"
        df_completo.to_csv(path_completo, index=False)

        ocr_ok   = (df_completo["correcao"] == "ocr_aceito").sum()
        snapped  = df_completo["correcao"].str.startswith("snap_").sum()
        fallback = df_completo["correcao"].str.startswith("fallback").sum()
        log.info("  Salvo: %s e %s", path_massa.name, path_completo.name)
        log.info("  OCR direto: %d | Snap: %d | Fallback: %d | Total: %d",
                 ocr_ok, snapped, fallback, len(df_completo))

    log.info("Concluído. CSVs em '%s'.", output_dir)


# Ver imagem do display para verificar se está enquadrada corretamente
def debug_tempo(video: str, tempo: float, roi: dict = ROI):
    cap = cv.VideoCapture(video)
    fps = cap.get(cv.CAP_PROP_FPS)
    cap.set(cv.CAP_PROP_POS_FRAMES, int(round(tempo * fps)))
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError("Não foi possível ler o frame.")
    y1, y2, x1, x2 = roi["y1"], roi["y2"], roi["x1"], roi["x2"]
    display = frame[y1:y2, x1:x2]
    cv.imwrite("debug_display.png", display)
    for nome, img in preprocessar(display):
        cv.imwrite(f"debug_{nome}.png", img)
    log.info("Debug salvo. Esperado para t=%.2fs: %.0fg", tempo, valor_esperado(tempo))

# Execução
if __name__ == "__main__":
    processar_todas()

14:05:22 [WARNING] Using CPU. Note: This module is much faster with a GPU.
14:05:26 [INFO] === Amostra 1 | Vídeo: video_1.mp4 | CSV: VariacaoRelativa_Amostra1.csv ===
14:05:26 [INFO]   Vídeo: video_1.mp4 | FPS: 29.96 | Pontos: 130
C:\Users\gisela25026\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
14:11:41 [INFO]   [20/130] t=10.63s → ocr=0.0 | final=100.0 | fallback_esperado
14:17:51 [INFO]   [40/130] t=21.10s → ocr=0.0 | final=200.0 | fallback_esperado
14:23:50 [INFO]   [60/130] t=31.58s → ocr=0.0 | final=300.0 | fallback_esperado
14:30:28 [INFO]   [80/130] t=42.06s → ocr=0.0 | final=400.0 | fallback_esperado
14:36:46 [INFO]   [100/130] t=52.54s → ocr=0.0 | final=500.0 | fallback_esperado
14:43:10 [INFO]   [120/130] t=63.02s → ocr=0.0 | final=500.0 | fallback_esperado
14:46:25 [INFO]   [130/130] t=68.92s